In [2]:
import boto3

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import get_execution_role
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output
)

region = boto3.Session().region_name
role = get_execution_role()

bucket = "krushang-beverage-ml-2026"

input_s3_uri = (
    f"s3://{bucket}/raw/survey_results.csv"
)

output_s3_uri = (
    f"s3://{bucket}/processed/"
)

script_path = (
    "/home/sagemaker-user/"
    "Beverage-Price-Prediction-AWS/"
    "src/preprocessing/preprocess.py"
)

print("Region:", region)
print("Execution role:", role)
print("Input:", input_s3_uri)
print("Output:", output_s3_uri)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: ap-south-1
Execution role: arn:aws:iam::812224290846:role/service-role/AmazonSageMaker-ExecutionRole-20260902T184223
Input: s3://krushang-beverage-ml-2026/raw/survey_results.csv
Output: s3://krushang-beverage-ml-2026/processed/


In [6]:
sklearn_image_uri = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2",
    py_version="py3",
    instance_type="ml.t3.medium"
)

In [7]:
processor = ScriptProcessor(
    image_uri=sklearn_image_uri,
    role=role,
    instance_type="ml.t3.medium",
    instance_count=1,
    base_job_name="beverage-preprocessing"
)

In [8]:
processor.run(
    code=script_path,

    inputs=[
        ProcessingInput(
            input_name="raw-data",
            s3_input=ProcessingS3Input(
                s3_uri=input_s3_uri,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File"
            )
        )
    ],

    outputs=[
        ProcessingOutput(
            output_name="processed-data",
            s3_output=ProcessingS3Output(
                s3_uri=output_s3_uri,
                local_path="/opt/ml/processing/output",
                s3_upload_mode="EndOfJob"
            )
        )
    ],

    wait=True,
    logs=True
)

[09/03/26 16:20:46] INFO     Creating processing-job with name                                    ]8;id=4580384;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/processing.py\processing.py]8;;\:]8;id=4580385;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/processing.py#617\617]8;;\
                             beverage-preprocessing-2026-09-03-16-20-46-539                                        

Output()

[09/03/26 16:20:47] WARNING  No region provided. Using default region.                                 ]8;id=4580392;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=4580393;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

[09/03/26 16:22:43] INFO     beverage-preprocessing-2026-09-03-16-20-46-539/algo-1-1788452485:   ]8;id=4580400;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4580401;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#29093\29093]8;;\
                             {                                                                                     
                                 "rows_raw": 30010,                                                                
                                 "duplicate_rows_removed": 10,                                                     
                                 "logical_outliers_removed": 44,                                                   
                                 "rows_clean": 29956,                                                              
                                 "columns_clean": 18,                                                              
                                 "remaining_missing": {                                                            
                                     "consume_frequency(weekly)": 8,                                               
                                     "purchase_channel": 10                                                        
                                 }                                                                                 

                    INFO     beverage-preprocessing-2026-09-03-16-20-46-539/algo-1-1788452485:   ]8;id=4580406;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4580407;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#29093\29093]8;;\
                             }                                                                                     

                    INFO     beverage-preprocessing-2026-09-03-16-20-46-539/algo-1-1788452485:   ]8;id=4580412;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4580413;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#29093\29093]8;;\
                             Saved cleaned data to:                                                                
                             /opt/ml/processing/output/cleaned_survey_results.csv                                  

[09/03/26 16:23:18] INFO     Final Resource Status: Completed                                    ]8;id=4580419;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4580420;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#29099\29099]8;;\

In [9]:
import boto3
import json
import pandas as pd
from io import BytesIO

s3 = boto3.client("s3")
bucket = "krushang-beverage-ml-2026"

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix="processed/"
)

print("Objects in processed/:")
for obj in response.get("Contents", []):
    print(obj["Key"], obj["Size"])

Objects in processed/:
processed/ 0
processed/cleaned_survey_results.csv 5741597
processed/preprocessing_metadata.json 250


In [10]:
obj = s3.get_object(
    Bucket=bucket,
    Key="processed/cleaned_survey_results.csv"
)

processed_df = pd.read_csv(
    BytesIO(obj["Body"].read())
)

print("Processed S3 shape:", processed_df.shape)

print("\nRemaining missing:")
print(
    processed_df.isna().sum()[
        processed_df.isna().sum() > 0
    ]
)

Processed S3 shape: (29956, 18)

Remaining missing:
consume_frequency(weekly)     8
purchase_channel             10
dtype: int64
